<a href="https://colab.research.google.com/github/iamdavid2/data_science/blob/main/Copy_of_camelot_quickstart_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quickstart example

This notebook shows you how to quickly get started with [camelot](https://github.com/camelot-dev/camelot) .

**Usage:** Either upload PDFs or add a URL to a PDF in the specified cells.

In [ ]:
# @title 🛠️ Install [camelot](https://github.com/camelot-dev/camelot)
!pip install camelot-py
# install tabulate (optional) only needed in this notebook for pretty display of results.
!pip install tabulate

In [ ]:
# Bootstrap and common imports
import os, sys, time

sys.path.insert(0, os.path.abspath(""))  # Prefer the local version if available

import camelot

print(f"Using camelot v{camelot.__version__}.")

In [ ]:
# @title ⚙️ Core - Complex Tables (Loose Parameters) with Clean Output

# import os
# from pathlib import Path
import pandas as pd
import numpy as np
from tabulate import tabulate

# Create output directory if it doesn't exist
output_dir = Path("/content/output")
output_dir.mkdir(parents=True, exist_ok=True)

# Process all PDF files in the input directory
input_dir = Path("/content/sample_pdfs")
for pdf_file in input_dir.glob("*.pdf"):
    print(f"\nProcessing {pdf_file.name}")

    # Using 'network' parsing method with table_areas
    tables_network = camelot.read_pdf(str(pdf_file), flavor="network")

    if len(tables_network) == 0:
        # If no tables are detected, try using 'lattice' parser
        tables_lattice = camelot.read_pdf(
            str(pdf_file), flavor="lattice", table_areas=["50,750,500,50"]
        )

    # Checking the detected tables
    if len(tables_network) > 0:
        tables = tables_network
    elif len(tables_lattice) > 0:
        tables = tables_lattice
    else:
        tables = []

    # Exporting if tables are found
    if len(tables) > 0:
        output_base = output_dir / pdf_file.stem
        tables.export(
            f"{output_base}.csv", f="csv", compress=True
        )  # export all tables to CSV
        tables[0].to_csv(
            f"{output_base}_first_table.csv"
        )  # Save the first table to CSV
        df = tables[0].df  # Get the first table as a pandas DataFrame

        print(f"Tables found in {pdf_file.name}:")

        # Clean up the DataFrame
        df = df.applymap(
            lambda x: x.strip() if isinstance(x, str) else x
        )  # Remove leading/trailing whitespace
        df = df.replace(["", "nan", "NaN", "NULL"], np.nan).dropna(
            how="all"
        )  # Remove empty rows
        df = df.fillna("")  # Replace NaN with empty string for display
        df = df.reset_index(drop=True)  # Reset index after dropping rows
    else:
        print(f"No tables found in {pdf_file.name}")



# Usar primera fila como nombres de columnas
df.columns = df.iloc[0]

# Quitar la primera fila
df = df.iloc[1:].reset_index(drop=True)

df['$Final'] = df['$Final\nObservaciones'].str.extract(r'^(\d+\.\d{3})')
df['$Final\nObservaciones'] = df['$Final\nObservaciones'].str.replace(r'^\d+\.\d{3}\s*', '', regex=True).str.strip()
df['Cantidad'] = df['Sexo Can'].str.split('\n').str[1]
df['Sexo Can'] = df['Sexo Can'].str.split('\n').str[0]
df = df.rename(columns={'$Final\nObservaciones': 'Observaciones'})
df = df.rename(columns={'Sexo Can': 'Sexo'})
for i in df.index:
    if  str(df.loc[i, 'Lote']).strip() == '':
        df.loc[i-1, 'Observaciones'] = str(df.loc[i-1, 'Observaciones']) + ' ' + str(df.loc[i, 'Observaciones'])
df = df[df['Lote'] != ''].reset_index(drop=True)
df

print("\nProcessing complete. Check the output directory for results.")

In [ ]:
df